# 01 · Read flux CSVs and PWB tlag summaries → Parquet

First processing step. Each EddyPro flux output (FLUXNET CSV format, one file
per [processing version](../docs/time-lag.qmd)) is read with `diive`
and written back out as a Parquet file for fast, type-safe reloading in the
downstream analysis.

- **Input:** `data/00-eddypro_fluxes_level-1/` — the raw EddyPro FLUXNET CSVs.
- **Output:** `data/01-eddypro_fluxes_level-1_parquet/` — one `.parquet` per version.

The study compares five scenarios per analyzer (QCL and LGR), ten in total. All
five, `*-1` through `*-5`, now have flux output and are read here. The `*-5`
(PWB) scenario differs only in how its time lag was handled: the lag was detected
and removed from the raw data before flux processing, so the EddyPro run itself
used no time-lag compensation. Its fluxes are read here exactly like the others.
The per-chunk PWB time-lag summaries are converted separately in the last section.

The reader is the same one used in diive's example data
(`load_exampledata_EDDYPRO_FLUXNET_CSV_30MIN`): `ReadFileType` with
`filetype='EDDYPRO-FLUXNET-CSV-30MIN'`. Saving uses `diive.core.io.files.save_parquet`.

## Imports

In [1]:
import re
from datetime import datetime
from pathlib import Path

import pandas as pd

from diive.core.io.filereader import ReadFileType
from diive.core.io.files import save_parquet

NB_START = datetime.now()  # notebook start time (reported in the last cell)

## Configuration

In [2]:
# Input / output folders (relative to the notebooks/ directory).
INDIR = Path("../data/00-eddypro_fluxes_level-1")
OUTDIR = Path("../data/01-eddypro_fluxes_level-1_parquet")
OUTDIR.mkdir(parents=True, exist_ok=True)

# Discover the input CSVs. Each filename is prefixed with its scenario code
# (e.g. 'LGR-1_eddypro_..._adv.csv', 'QCL-4_2020_..._adv.csv').
csv_files = sorted(INDIR.glob("*_adv.csv"))
print(f"Found {len(csv_files)} flux file(s):")
for f in csv_files:
    print(f"  {f.name}")

Found 10 flux file(s):
  LGR-1_eddypro_LGR-1_FR-20260619-164852_fluxnet_2026-06-20T073939_adv.csv
  LGR-2_eddypro_LGR-2_FR-20260615-140834_fluxnet_2026-06-16T050657_adv.csv
  LGR-3_eddypro_LGR-3_FR-20260615-140917_fluxnet_2026-06-16T035745_adv.csv
  LGR-4_eddypro_LGR-4_FR-20260623-133726_fluxnet_2026-06-24T014358_adv.csv
  LGR-5_eddypro_LGR-5_FR-20260622-174418_fluxnet_2026-06-23T044426_adv.csv
  QCL-1_eddypro_QCL-1_FR-20260619-164952_fluxnet_2026-06-20T101745_adv.csv
  QCL-2_eddypro_QCL-2_FR-20260615-140229_fluxnet_2026-06-16T073700_adv.csv
  QCL-3_eddypro_QCL-3_FR-20260615-140311_fluxnet_2026-06-16T060134_adv.csv
  QCL-4_eddypro_QCL-4_FR-20260623-134019_fluxnet_2026-06-24T035207_adv.csv
  QCL-5_eddypro_QCL-5_FR-20260622-174805_fluxnet_2026-06-23T065147_adv.csv


## Version codes

In [3]:
def version_code(filename: str) -> str:
    """Extract the scenario code (e.g. 'LGR-1', 'QCL-4') from a filename.

    The code is the leading token of the filename, before the first underscore.
    """
    match = re.match(r"([A-Z]+-\d+)_", filename)
    if not match:
        raise ValueError(f"Could not extract a version code from: {filename}")
    return match.group(1)


# Map each input file to its version code (the output Parquet name).
for f in csv_files:
    print(f"{version_code(f.name):>8}  <-  {f.name}")

   LGR-1  <-  LGR-1_eddypro_LGR-1_FR-20260619-164852_fluxnet_2026-06-20T073939_adv.csv
   LGR-2  <-  LGR-2_eddypro_LGR-2_FR-20260615-140834_fluxnet_2026-06-16T050657_adv.csv
   LGR-3  <-  LGR-3_eddypro_LGR-3_FR-20260615-140917_fluxnet_2026-06-16T035745_adv.csv
   LGR-4  <-  LGR-4_eddypro_LGR-4_FR-20260623-133726_fluxnet_2026-06-24T014358_adv.csv
   LGR-5  <-  LGR-5_eddypro_LGR-5_FR-20260622-174418_fluxnet_2026-06-23T044426_adv.csv
   QCL-1  <-  QCL-1_eddypro_QCL-1_FR-20260619-164952_fluxnet_2026-06-20T101745_adv.csv
   QCL-2  <-  QCL-2_eddypro_QCL-2_FR-20260615-140229_fluxnet_2026-06-16T073700_adv.csv
   QCL-3  <-  QCL-3_eddypro_QCL-3_FR-20260615-140311_fluxnet_2026-06-16T060134_adv.csv
   QCL-4  <-  QCL-4_eddypro_QCL-4_FR-20260623-134019_fluxnet_2026-06-24T035207_adv.csv
   QCL-5  <-  QCL-5_eddypro_QCL-5_FR-20260622-174805_fluxnet_2026-06-23T065147_adv.csv


## Read each CSV and save as Parquet

`ReadFileType` returns the data (`data_df`) and a metadata table (`metadata_df`,
the variable units). The data frame is written to Parquet named after the
version code, so downstream code can load e.g. `LGR-1.parquet` directly.

In [4]:
saved = {}
for f in csv_files:
    code = version_code(f.name)
    print(f"\n=== {code} ===")

    loaddatafile = ReadFileType(
        filetype="EDDYPRO-FLUXNET-CSV-30MIN",
        filepath=str(f),
        data_nrows=None,
        output_middle_timestamp=True,
    )
    data_df, metadata_df = loaddatafile.get_filedata()
    print(f"  read {data_df.shape[0]} rows x {data_df.shape[1]} cols")

    filepath = save_parquet(filename=code, data=data_df, outpath=str(OUTDIR))
    saved[code] = filepath


=== LGR-1 ===


> Reading file LGR-1_eddypro_LGR-1_FR-20260619-164852_fluxnet_2026-06-20T073939_adv.csv ...

F:\dev\diive\diive\core\io\filereader.py:591: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[_temp_parsed_index_col] = ''


  read 7805 rows x 484 cols


> Saved file ..\data\01-eddypro_fluxes_level-1_parquet\LGR-1.parquet (0.192 seconds).

...


=== QCL-5 ===


> Reading file QCL-5_eddypro_QCL-5_FR-20260622-174805_fluxnet_2026-06-23T065147_adv.csv ...

F:\dev\diive\diive\core\io\filereader.py:591: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[_temp_parsed_index_col] = ''


  read 9642 rows x 482 cols


> Saved file ..\data\01-eddypro_fluxes_level-1_parquet\QCL-5.parquet (0.371 seconds).

## Verify

Reload one Parquet file to confirm the round-trip and inspect a few of the
study-relevant columns (fluxes + time-lag diagnostics).

**Table.** Summary statistics of the study-relevant columns after the Parquet
round trip (LGR-1): the two fluxes and their time-lag diagnostics.

In [5]:
from diive.core.io.files import load_parquet

print("Saved Parquet files:")
for code, path in saved.items():
    print(f"  {code}: {path}")

check = load_parquet(filepath=saved["LGR-1"])
cols = [c for c in ["FN2O", "FCH4", "N2O_TLAG_USED", "CH4_TLAG_USED"] if c in check.columns]
print(f"\nLGR-1: {check.shape[0]} rows x {check.shape[1]} cols")
check[cols].describe()

Saved Parquet files:
  LGR-1: ..\data\01-eddypro_fluxes_level-1_parquet\LGR-1.parquet
  LGR-2: ..\data\01-eddypro_fluxes_level-1_parquet\LGR-2.parquet
  LGR-3: ..\data\01-eddypro_fluxes_level-1_parquet\LGR-3.parquet
  LGR-4: ..\data\01-eddypro_fluxes_level-1_parquet\LGR-4.parquet
  LGR-5: ..\data\01-eddypro_fluxes_level-1_parquet\LGR-5.parquet
  QCL-1: ..\data\01-eddypro_fluxes_level-1_parquet\QCL-1.parquet
  QCL-2: ..\data\01-eddypro_fluxes_level-1_parquet\QCL-2.parquet
  QCL-3: ..\data\01-eddypro_fluxes_level-1_parquet\QCL-3.parquet
  QCL-4: ..\data\01-eddypro_fluxes_level-1_parquet\QCL-4.parquet
  QCL-5: ..\data\01-eddypro_fluxes_level-1_parquet\QCL-5.parquet


> Loaded .parquet file ..\data\01-eddypro_fluxes_level-1_parquet\LGR-1.parquet (0.070 seconds).


LGR-1: 7805 rows x 484 cols


,FN2O,FCH4,N2O_TLAG_USED,CH4_TLAG_USED
count,7731.000000,7731.000000,7731.000000,7731.000000
mean,1.835840,15.371666,3.644225,4.451856
std,9.187183,221.542376,3.104364,3.659461
min,-374.774000,-7207.160000,-0.050000,-0.050000
25%,0.115590,-11.456300,1.650000,1.650000
50%,0.725180,4.458190,1.950000,2.750000
75%,2.551030,26.860500,5.950000,8.400000
max,205.036000,5272.880000,10.000000,10.000000


## PWB tlag summaries

The `*-5` PWB scenario (`LGR-5`, `QCL-5`) now has flux output (read above with
the other scenarios). In addition, its per-chunk time-lag results are available
(see [processing steps](../docs/time-lag.qmd)). Their summary CSVs in
`data/00-pwb_tlag_summary/` are plain tables (one row per processed chunk), so
they are read with pandas rather than `ReadFileType`.

Only chunks whose start is on the 30-minute wall-clock grid are kept. The leading
partial chunks (off-grid starts like 10:10) are dropped so the series aligns to
the same regular timestamp grid as the flux data; otherwise `load_parquet` would
reindex them onto the grid and null their values.

The result is written to a separate folder
(`data/01-pwb_tlag_summary_parquet/`) with a `pwb_tlag` tag in the filename
(e.g. `LGR-5_pwb_tlag.parquet`), so it stays distinct from the flux Parquets and
is not confused with the flux tables downstream.

In [6]:
PWB_INDIR = Path("../data/00-pwb_tlag_summary")
PWB_OUTDIR = Path("../data/01-pwb_tlag_summary_parquet")
PWB_OUTDIR.mkdir(parents=True, exist_ok=True)

pwb_files = sorted(PWB_INDIR.glob("*_detect_and_remove_tlag_summary.csv"))
print(f"Found {len(pwb_files)} PWB summary file(s):")

pwb_saved = {}
for f in pwb_files:
    code = re.match(r"([A-Z]+-\d+)_", f.name).group(1)  # e.g. 'LGR-5'
    df = pd.read_csv(f, parse_dates=["timestamp"]).set_index("timestamp")

    # Keep only chunks whose start is on the 30-min wall-clock grid. The leading
    # partial chunks (e.g. 10:10) are off-grid and would not align to the regular
    # timestamp grid used by the flux data, so they are dropped.
    on_grid = df.index.minute.isin([0, 30]) & (df.index.second == 0)
    n_off = int((~on_grid).sum())
    df = df[on_grid]

    # The PWB timestamp is the chunk START time; name the index so diive's
    # load_parquet accepts it.
    df.index.name = "TIMESTAMP_START"
    print(f"  {code}: {df.shape[0]} rows x {df.shape[1]} cols ({n_off} off-grid chunks dropped)")
    filepath = save_parquet(filename=f"{code}_pwb_tlag", data=df, outpath=str(PWB_OUTDIR))
    pwb_saved[code] = filepath

Found 2 PWB summary file(s):
  LGR-5: 7805 rows x 62 cols (15 off-grid chunks dropped)


> Saved file ..\data\01-pwb_tlag_summary_parquet\LGR-5_pwb_tlag.parquet (0.036 seconds).

  QCL-5: 9520 rows x 62 cols (8 off-grid chunks dropped)


> Saved file ..\data\01-pwb_tlag_summary_parquet\QCL-5_pwb_tlag.parquet (0.050 seconds).

## Runtime

In [7]:
NB_END = datetime.now()
print(f"Start:    {NB_START:%Y-%m-%d %H:%M:%S}")
print(f"End:      {NB_END:%Y-%m-%d %H:%M:%S}")
print(f"Runtime:  {NB_END - NB_START}")

Start:    2026-06-24 15:15:41
End:      2026-06-24 15:16:12
Runtime:  0:00:30.562765
